# Image Embedding Model Evaluation

This experiment evaluates the performance of pretrained image embedding models trained on the ImageNet dataset under a CPU-only environment. The objective is to analyse the trade-off between identification accuracy and computational efficiency in the context of object similarity matching. Specifically, the study compares multiple pretrained architectures by measuring Top-1 identification accuracy, embedding inference time, and overall end-to-end pipeline latency. The goal is to determine the most suitable model for the proposed object retrieval system, balancing accurate instance matching with practical CPU execution constraints.

The evaluation will be based on 
- cosine similarity
- Top-1 Identification Accuracy
- Top-5 Retrieval Accuracy
- Latency
- End-to-end Processing Time

Top-1 identification accuracy measures the percentage of queries where the correct object instance is retrieved as the most similar result. It reflects the model’s ability to rank the true match highest among all candidates. Top-5 retrieval accuracy extends this by measuring whether the correct instance appears within the five most similar results, providing insight into the model’s broader ranking capability even if the correct match is not ranked first. Together, these metrics evaluate how effectively the embedding model supports reliable object similarity matching.

The models that will be evaluated include:

- CLIP ViT-B/32
- ResNet-50 (ImageNet pretrained)
- MobileNetV3 (ImageNet pretrained)

The YOLOv8n model was selected as the preferred object detection model from the object detection evaluation jupyter notebook and executed to generate cropped regions from input images. These crops simulate the actual deployment scenario, where object-level regions are required rather than full images. Image embeddings are then extracted from each cropped region and compared against a reference set of images to compute evaluation metrics, enabling assessment of the embedding models performance.

In [ ]:
from ultralytics import YOLO
import torch

# setting the device to CPU
device = "cpu"
# loading the YOLO pretriained model
model = YOLO("yolov8n.pt")
model.to(device)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

A quick warm up is done to ensure that the subsequent inferences run more consistently.

In [ ]:
import numpy as np
# dummy image created
dummy = np.zeros((640, 640, 3), dtype=np.uint8)
# running through the model for a warm up
_ = model(dummy, device=device, verbose=False)

The object detection model is applied to each image to extract the highest-confidence detected object. Since the primary objective is to generate cropped inputs for the embedding model, perfect detection accuracy is not critical. Minor inaccuracies in cropping are acceptable, as they do not significantly impact the overall pipeline. Any clearly incorrect crops that fail to capture the object will be manually reviewed and corrected.

The dataset consists of 8 objects, each represented by a pair of images taken in different settings. After cropping the detected objects, the images are converted into vector embeddings using an embedding model. These embeddings are then compared to assess how well the model can identify matching objects

In [ ]:
import os
import cv2
import numpy as np

input_root = "photos"
# cropped images will be saved in a seperate folder
output_root = "cropped_photos"
os.makedirs(output_root, exist_ok=True)

for object_folder in os.listdir(input_root):

    object_path = os.path.join(input_root, object_folder)
    # if it is not a folder skip it
    if not os.path.isdir(object_path):
        continue

    # Create corresponding output folder
    output_object_path = os.path.join(output_root, object_folder)
    os.makedirs(output_object_path, exist_ok=True)

    for filename in os.listdir(object_path):

        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(object_path, filename)
        # reading the image using opencv
        image = cv2.imread(img_path)

        if image is None:
            print(f"Failed to read {img_path}")
            continue
        # running object detection
        results = model(image, device=device, verbose=False)
        # extracting bounding boxes
        boxes = results[0].boxes

        if boxes is None or len(boxes) == 0:
            print(f"No detection for {img_path}")
            continue

        # Choose highest confidence detection
        confidences = boxes.conf.cpu().numpy()
        # get the index of the highest confidence box
        best_idx = np.argmax(confidences)
        # get bounding box coordinates
        xyxy = boxes.xyxy.cpu().numpy()[best_idx]
        x1, y1, x2, y2 = map(int, xyxy)

        # Clamp coordinates to stay within image boundaries
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(image.shape[1], x2)
        y2 = min(image.shape[0], y2)
        # cropping the detected region
        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            print(f"Invalid crop for {img_path}")
            continue

        save_path = os.path.join(output_object_path, filename)
        # saving image
        cv2.imwrite(save_path, crop)

        print(f"Cropped and saved: {save_path}")

print("Cropping complete.")

Cropped and saved: cropped_photos\object_1\img1.jpeg
Cropped and saved: cropped_photos\object_1\img2.jpeg
Cropped and saved: cropped_photos\object_2\img1.jpeg
Cropped and saved: cropped_photos\object_2\img2.jpeg
Cropped and saved: cropped_photos\object_3\img1.jpeg
Cropped and saved: cropped_photos\object_3\img2.jpeg
Cropped and saved: cropped_photos\object_4\img1.jpeg
No detection for photos\object_4\img2.jpeg
Cropped and saved: cropped_photos\object_5\img1.jpeg
Cropped and saved: cropped_photos\object_5\img2.jpeg
Cropped and saved: cropped_photos\object_6\img1.jpeg
Cropped and saved: cropped_photos\object_6\img2.jpeg
No detection for photos\object_7\img1.jpeg
No detection for photos\object_7\img2.jpeg
No detection for photos\object_8\img1.jpeg
Cropped and saved: cropped_photos\object_8\img2.jpeg
Cropping complete.


## CLIP

The pretrained CLIP model is first loaded

In [ ]:
from transformers import CLIPProcessor, CLIPModel
# importing CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_model = clip_model.to(device)
clip_model.eval()
# processor is loaded for image preprocesssing and tokenization
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

c:\Users\gohji\anaconda3\envs\ObjectDetection-FYP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 435.94it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking cha

Extract embedding function is created to process the list of images and then generate a normalized vector embedding using the CLIP vision encoder.

In [ ]:
import time
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

def extract_embeddings(image_paths):
    # storing embedding and inference times
    embeddings = []
    inference_times = []

    # Warmup for model initialization to avoid first run latency
    dummy_img = Image.open(image_paths[0]).convert("RGB")
    dummy_inputs = clip_processor(images=dummy_img, return_tensors="pt")
    pixel_values = dummy_inputs["pixel_values"].to(device)
    # running a dummy foward pass
    with torch.no_grad():
        vision_out = clip_model.vision_model(pixel_values=pixel_values)
        image_features = vision_out.pooler_output
        image_features = clip_model.visual_projection(image_features)
    # processing each image and extract embeddings
    with torch.no_grad():
        for img_path in tqdm(image_paths):
            # loading and preprocessing image
            image = Image.open(img_path).convert("RGB")
            inputs = clip_processor(images=image, return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)
            # inference timer
            start = time.perf_counter()
            # foward pass through the CLIP vision encoder
            vision_out = clip_model.vision_model(pixel_values=pixel_values)
            image_features = vision_out.pooler_output
            image_features = clip_model.visual_projection(image_features)

            end = time.perf_counter()
            inference_times.append(end - start)
            # normalize embeddings so similarity comparison can be performed
            image_features = image_features / torch.norm(image_features, dim=-1, keepdim=True)
            # storing embeddings imto numpy array
            embeddings.append(image_features.cpu().numpy())
    # Stack all image embeddings into one array for easier comparison
    embeddings = np.vstack(embeddings)
    avg_latency = float(np.mean(inference_times))

    return embeddings, avg_latency

Before running the embedding model, the dataset is organized into gallery and query sets for evaluation. Each object folder is assigned a unique label, with one image placed in the gallery and the other in the query set. This pairing allows embeddings from query images to be compared against the gallery to measure how well the model matches the same object across different images.

In [ ]:
import os
import cv2
import numpy as np

# list to store gallery image paths and labels
gallery_paths = []
gallery_labels = []

# list to store image paths and their labels
query_paths = []
query_labels = []

root_dir = "cropped_photos"
# looping through each object folder and assigning a label index
for label_idx, folder in enumerate(sorted(os.listdir(root_dir))):
    folder_path = os.path.join(root_dir, folder)

    if not os.path.isdir(folder_path):
        continue
    # define expected image pair names
    img1 = os.path.join(folder_path, "img1.jpeg")
    img2 = os.path.join(folder_path, "img2.jpeg")
    # if both images exist, add first image to the gallery set and then add the second image to the query set
    if os.path.exists(img1) and os.path.exists(img2):
        gallery_paths.append(img1)
        gallery_labels.append(label_idx)

        query_paths.append(img2)
        query_labels.append(label_idx)
# print dataset size to ensure that the number of images in each set is correct
print("Gallery size:", len(gallery_paths))
print("Query size:", len(query_paths))

Gallery size: 8
Query size: 8


The embedding pipeline will then be ran. The embeddings will be extracted for both the gallery and query images, then computing a similarity matrix between them. The evaluation metrics as stated above will also be calculated

In [ ]:
# end to end time calculation
start_total = time.perf_counter()
# extract embeddins for gallery images and record the average latency
gallery_embeddings, gallery_latency = extract_embeddings(gallery_paths)
# extract embeddins for query images and record the average latency
query_embeddings, query_latency = extract_embeddings(query_paths)
# Compute pairwise cosine similarity between query and gallery embeddings
# Result is a matrix where each row corresponds to a query and each column to a gallery image
# cosine similarity is calculated by the matrix multiplication of normalized vectors
similarity_matrix = np.matmul(query_embeddings, gallery_embeddings.T)

end_total = time.perf_counter()

end_to_end_time = end_total - start_total

100%|██████████| 8/8 [00:00<00:00, 13.34it/s]


In [ ]:
top1_correct = 0
top5_correct = 0
true_similarities = []

for i in range(len(query_labels)):
    # getting the similarity scores between current query and all gallery images
    sims = similarity_matrix[i]
    # ranking indices by descending similarity
    ranked_indices = np.argsort(-sims)

    # checking top 1 accuracy
    if gallery_labels[ranked_indices[0]] == query_labels[i]:
        top1_correct += 1

    # Top-5 accuracy
    top5 = ranked_indices[:5]
    if query_labels[i] in [gallery_labels[idx] for idx in top5]:
        top5_correct += 1

    # store the cosine similarity of the correct match
    true_similarities.append(sims[query_labels[i]])
# computing final accuracy metrics and average similarity
top1_acc = top1_correct / len(query_labels)
top5_acc = top5_correct / len(query_labels)
mean_cosine_similarity = np.mean(true_similarities)

print("Top-1 Accuracy:", top1_acc)
print("Top-5 Accuracy:", top5_acc)
print("Mean Cosine Similarity:", mean_cosine_similarity)
print("Average Inference Latency (gallery):", gallery_latency)
print("Average Inference Latency (query):", query_latency)
print("End-to-End Time:", end_to_end_time)

Top-1 Accuracy: 0.625
Top-5 Accuracy: 1.0
Mean Cosine Similarity: 0.83733994
Average Inference Latency (gallery): 0.05649319999974978
Average Inference Latency (query): 0.058168212503005634
End-to-End Time: 1.4802817000017967


Store data in a pandas dataframe for easier analysis later on

In [9]:
import pandas as pd
results = []

results.append({
    "Model": "CLIP ViT-B/32",
    "Top1_Accuracy": top1_acc,
    "Top5_Accuracy": top5_acc,
    "Mean_Cosine_Similarity": mean_cosine_similarity,
    "Gallery_Latency_sec": gallery_latency,
    "Query_Latency_sec": query_latency,
    "End_to_End_Time_sec": end_to_end_time
})

In [10]:
results_df = pd.DataFrame(results)
results_df

,Model,Top1_Accuracy,Top5_Accuracy,Mean_Cosine_Similarity,Gallery_Latency_sec,Query_Latency_sec,End_to_End_Time_sec
0,CLIP ViT-B/32,0.625,1.0,0.83734,0.056493,0.058168,1.480282


## ResNet-50

This code initializes a pre-trained ResNet-50 model and modifies it into a feature extractor by removing its final classification layer. The classification layer is not needed because the model is used to generate vector embeddings for similarity comparison rather than predicting predefined classes.

In [11]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import time
from tqdm import tqdm

device = "cpu"

# Load pretrained ResNet-50
resnet_model = models.resnet50(pretrained=True)

# Remove final classification layer
resnet_model = torch.nn.Sequential(*list(resnet_model.children())[:-1])

resnet_model.eval()
resnet_model.to(device)

c:\Users\gohji\anaconda3\envs\ObjectDetection-FYP\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\gohji\anaconda3\envs\ObjectDetection-FYP\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\gohji/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 63.0MB/s]


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


Before passing images into the ResNet model, they must be preprocessed to match the format the model was trained on. Without this step, the model would produce unreliable embeddings.

This code defines a preprocessing pipeline that resizes images to 224×224, converts them into tensors, and normalizes them using ImageNet statistics to ensure consistency with the model’s training conditions.

In [ ]:
resnet_transform = transforms.Compose([
    # Resize image to 224x224 (required input size for ResNet)
    transforms.Resize((224, 224)),
    # Convert image to PyTorch tensor (scales pixel values to [0,1])
    transforms.ToTensor(),
    # normalizing  to match how resnet was originally trained
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

This function generates vector embeddings for a set of images using a modified ResNet-50 model.

In [ ]:
def extract_resnet_embeddings(image_paths):
    embeddings = []
    inference_times = []

    # Warmup to initialise model and avoid first run latency
    dummy_img = Image.open(image_paths[0]).convert("RGB")
    dummy_tensor = resnet_transform(dummy_img).unsqueeze(0).to(device)
    # dummy foward pass
    with torch.no_grad():
        _ = resnet_model(dummy_tensor)

    with torch.no_grad():
        for img_path in tqdm(image_paths):
            # load and process the image
            image = Image.open(img_path).convert("RGB")
            tensor = resnet_transform(image).unsqueeze(0).to(device)
            # inference timing calculation
            start = time.perf_counter()
            # foward pass through resnet
            features = resnet_model(tensor)
            end = time.perf_counter()

            inference_times.append(end - start)

            # flatten output from [1,2048,1,1] to [1,2048]
            features = features.view(features.size(0), -1)

            # normalize embedding for cosine similarity
            features = features / torch.norm(features, dim=-1, keepdim=True)
            # store embedding as numpy array
            embeddings.append(features.cpu().numpy())
    # stacking all embeddings into a single 2D arary
    embeddings = np.vstack(embeddings)
    # average inference latency computation
    avg_latency = float(np.mean(inference_times))

    return embeddings, avg_latency

The embedding pipeline will then be ran. The embeddings will be extracted for both the gallery and query images, then computing a similarity matrix between them. The evaluation metrics as stated above will also be calculated

In [ ]:

start_total = time.perf_counter()
# extract embeddings for gallery images using ResNet and record latency
gallery_embeddings_resnet, gallery_latency_resnet = extract_resnet_embeddings(gallery_paths)
# extract embeddings for query images using ResNet and record latency
query_embeddings_resnet, query_latency_resnet = extract_resnet_embeddings(query_paths)
# compute similarity matrix between query and gallery embeddings
similarity_matrix_resnet = query_embeddings_resnet @ gallery_embeddings_resnet.T

end_total = time.perf_counter()
# end to end time calculation
end_to_end_time_resnet = end_total - start_total

100%|██████████| 8/8 [00:00<00:00, 19.20it/s]


In [ ]:
top1_correct = 0
top5_correct = 0
true_similarities = []
# looping through each query image
for i in range(len(query_labels)):
    # get similarity scores between current query and all gallery images
    sims = similarity_matrix_resnet[i]
    # ranking based on descending similarity
    ranked_indices = np.argsort(-sims)

    # checking the top 1 accuracy
    if gallery_labels[ranked_indices[0]] == query_labels[i]:
        top1_correct += 1

    # checking the top 5 accuracy
    top5 = ranked_indices[:5]
    if query_labels[i] in [gallery_labels[idx] for idx in top5]:
        top5_correct += 1

    # store cosine similarity of correct match
    true_similarities.append(sims[query_labels[i]])
# compute final accuracy metrics and the average cosine similarity for the correct matches
top1_acc_resnet = top1_correct / len(query_labels)
top5_acc_resnet = top5_correct / len(query_labels)
mean_cosine_resnet = np.mean(true_similarities)

print("ResNet-50 Results")
print("Top-1 Accuracy:", top1_acc_resnet)
print("Top-5 Accuracy:", top5_acc_resnet)
print("Mean Cosine Similarity:", mean_cosine_resnet)
print("Average Inference Latency (gallery):", gallery_latency_resnet)
print("Average Inference Latency (query):", query_latency_resnet)
print("End-to-End Time:", end_to_end_time_resnet)

ResNet-50 Results
Top-1 Accuracy: 0.875
Top-5 Accuracy: 1.0
Mean Cosine Similarity: 0.78523767
Average Inference Latency (gallery): 0.0488898750008957
Average Inference Latency (query): 0.04576950000227953
End-to-End Time: 1.0452528000023449


Appending the results into the dataframe

In [16]:
results.append({
    "Model": "ResNet-50",
    "Top1_Accuracy": top1_acc_resnet,
    "Top5_Accuracy": top5_acc_resnet,
    "Mean_Cosine_Similarity": mean_cosine_resnet,
    "Gallery_Latency_sec": gallery_latency_resnet,
    "Query_Latency_sec": query_latency_resnet,
    "End_to_End_Time_sec": end_to_end_time_resnet
})

results_df = pd.DataFrame(results)
results_df

,Model,Top1_Accuracy,Top5_Accuracy,Mean_Cosine_Similarity,Gallery_Latency_sec,Query_Latency_sec,End_to_End_Time_sec
0,CLIP ViT-B/32,0.625,1.0,0.837340,0.056493,0.058168,1.480282
1,ResNet-50,0.875,1.0,0.785238,0.048890,0.045770,1.045253


## MobileNetv3

The pre-trained MobileNetV3-Large model is initialized and configured as a feature extractor for generating image embeddings.

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms

device = "cpu"
# loading pretrained mobilenet model
mobilenet_model = models.mobilenet_v3_large(pretrained=True)

# Remove classifier
mobilenet_model.classifier = torch.nn.Identity()

mobilenet_model.eval()
mobilenet_model.to(device)

c:\Users\gohji\anaconda3\envs\ObjectDetection-FYP\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\gohji\anaconda3\envs\ObjectDetection-FYP\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to C:\Users\gohji/.cache\torch\hub\checkpoints\mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 49.0MB/s]


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

Before passing images into the MobileNet model, they must be preprocessed to match the format used during training. This ensures that the extracted embeddings are consistent and reliable.

In [ ]:
# preprocessing pipeline for mobilenet input
mobilenet_transform = transforms.Compose([
    # Resize image, convert to tensor, and normalize using ImageNet statistics to match the model’s training conditions
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

This function extracts normalized vector embeddings from images using the MobileNetV3-Large model. It includes a warmup step to stabilize inference performance, processes each image while measuring latency, and aggregates all embeddings into a single array for similarity comparison and evaluation.

In [ ]:
import time
import numpy as np
from PIL import Image
from tqdm import tqdm

# function to extract embeddings using MobileNet
def extract_mobilenet_embeddings(image_paths):
    # store embeddings and inference times
    embeddings = []
    inference_times = []

    # warmup step to initialize model and avoid first run latency
    dummy_img = Image.open(image_paths[0]).convert("RGB")
    dummy_tensor = mobilenet_transform(dummy_img).unsqueeze(0).to(device)
    # run a dummy foward pass
    with torch.no_grad():
        _ = mobilenet_model(dummy_tensor)

    with torch.no_grad():
        for img_path in tqdm(image_paths):
            # loading and processing the image
            image = Image.open(img_path).convert("RGB")
            tensor = mobilenet_transform(image).unsqueeze(0).to(device)
            # inference timer start
            start = time.perf_counter()
            # foward pass through mobilenet
            features = mobilenet_model(tensor)
            # end timer
            end = time.perf_counter()

            inference_times.append(end - start)

            # normalize embeddings for cosine similarity later on
            features = features / torch.norm(features, dim=-1, keepdim=True)
            # storing of embeddings in an array
            embeddings.append(features.cpu().numpy())
    # combine all embedddings into a single 2D array
    embeddings = np.vstack(embeddings)
    avg_latency = float(np.mean(inference_times))

    return embeddings, avg_latency

The embedding pipeline will then be ran. The embeddings will be extracted for both the gallery and query images, then computing a similarity matrix between them. The evaluation metrics as stated above will also be calculated

In [ ]:
start_total = time.perf_counter()
# extract embeddings for gallery images using MobileNet and record latency
gallery_embeddings_mobile, gallery_latency_mobile = extract_mobilenet_embeddings(gallery_paths)
# extract embeddings for query images using MobileNet and record latency
query_embeddings_mobile, query_latency_mobile = extract_mobilenet_embeddings(query_paths)

# compute similarity matrix between query and gallery embeddings
similarity_matrix_mobile = query_embeddings_mobile @ gallery_embeddings_mobile.T

end_total = time.perf_counter()
# end to end time calculation
end_to_end_time_mobile = end_total - start_total

100%|██████████| 8/8 [00:00<00:00, 50.89it/s]


In [ ]:
top1_correct = 0
top5_correct = 0
true_similarities = []

for i in range(len(query_labels)):
    # get similarity scores between current query and all gallery images
    sims = similarity_matrix_mobile[i]
    # rank gallery indices by descending similarity
    ranked_indices = np.argsort(-sims)
    # check top 1 accuracy
    if gallery_labels[ranked_indices[0]] == query_labels[i]:
        top1_correct += 1
    # check top 5 accuracy
    top5 = ranked_indices[:5]
    if query_labels[i] in [gallery_labels[idx] for idx in top5]:
        top5_correct += 1
    # storing cosine similarity score
    true_similarities.append(sims[query_labels[i]])
# computing final top 1 and top 5 accuracy and average cosine similarity
top1_acc_mobile = top1_correct / len(query_labels)
top5_acc_mobile = top5_correct / len(query_labels)
mean_cosine_mobile = np.mean(true_similarities)


Appending results into the dataframe

In [22]:
results.append({
    "Model": "MobileNetV3",
    "Top1_Accuracy": top1_acc_mobile,
    "Top5_Accuracy": top5_acc_mobile,
    "Mean_Cosine_Similarity": mean_cosine_mobile,
    "Gallery_Latency_sec": gallery_latency_mobile,
    "Query_Latency_sec": query_latency_mobile,
    "End_to_End_Time_sec": end_to_end_time_mobile
})

In [23]:
results_df = pd.DataFrame(results)
results_df

,Model,Top1_Accuracy,Top5_Accuracy,Mean_Cosine_Similarity,Gallery_Latency_sec,Query_Latency_sec,End_to_End_Time_sec
0,CLIP ViT-B/32,0.625,1.0,0.837340,0.056493,0.058168,1.480282
1,ResNet-50,0.875,1.0,0.785238,0.048890,0.045770,1.045253
2,MobileNetV3,1.000,1.0,0.682849,0.016365,0.014377,0.423325


Based on the experimental evaluation, MobileNetV3 was selected as the optimal model for deployment in the object locator application. It achieved the highest Top-1 identification accuracy of 1.000 and maintained perfect Top-5 retrieval accuracy, outperforming both ResNet-50 and CLIP ViT-B/32 in instance-level matching on the tested dataset. In addition to its superior accuracy, MobileNetV3 demonstrated significantly lower computational latency, with an average gallery inference time of 0.016 seconds and an end-to-end processing time of 0.423 seconds, which is substantially faster than the other models. Given that the application is designed for CPU-only mobile deployment, computational efficiency and responsiveness are critical factors. MobileNetV3 provides the best balance between accuracy and speed, making it the most suitable model for real-time object retrieval in a resource-constrained mobile environment.